# NB01: Data Preparation

Load, join, validate, and encode the three raw data sources.
Produces `outputs/nb_model_ready.csv` — the single clean dataset
consumed by NB02, NB03, and NB04.

**Run this notebook first, and re-run it every time source data changes.**
Update source files first: `python scripts/update_data.py`

## Inputs / Outputs

| Direction | File | Description |
|-----------|------|-------------|
| Input | `data/raw/crash_counts.csv` | BRON 2019-2024: injury crash count + exposure per intersection |
| Input | `data/raw/intersection_features.csv` | NWB + traffic: infrastructure variables |
| Input | `data/raw/cv_scores_full_network.csv` | CV pipeline NB05: predicted safety score + cv_split label |
| **Output** | **`data/processed/nb_model_ready.csv`** | Joined, validated, encoded dataset for all downstream notebooks |
| Output | `data/processed/nb_model_ready_imputed.csv` | Sensitivity version: missing CV scores mean-imputed (see Section 4) |

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ROOT is one level up from pipeline/ where this notebook lives.
# If Jupyter sets cwd to the notebook directory, Path("..") resolves correctly.
ROOT      = Path("..").resolve()
DATA      = ROOT / "data" / "raw"
OUTPUTS   = ROOT / "outputs"              # final deliverables only
PROCESSED = ROOT / "data" / "processed"  # intermediate files passed between notebooks

OUTPUTS.mkdir(exist_ok=True)
PROCESSED.mkdir(exist_ok=True)

print(f"Project root : {ROOT}")
print(f"Data (raw)   : {DATA}")
print(f"Processed    : {PROCESSED}")
print(f"Outputs      : {OUTPUTS}")

## 1. Load Raw Sources

Three CSV files share the join key `intersection_id` (= NWB `JTE_ID`).

**Expected schemas:**

`crash_counts.csv`
- `intersection_id` (int) — NWB junction ID
- `crash_count` (int) — injury crashes 2019-2024
- `exposure` (float) — single number per intersection; units TBD

`intersection_features.csv`
- `intersection_id`, `n_legs`, `speed_limit`, `dim_type` (T / 4+), `dim_priority` (VRI / voorrang / geen_voorrang), `intensity_major`, `intensity_minor`

`cv_scores_full_network.csv`
- `intersection_id`, `cv_score_predicted` (float 0-1), `cv_split` (train / val / unlabelled / excluded)

In [10]:
# Load each raw source
crashes  = pd.read_csv(DATA / "crash_counts.csv",
                       dtype={"intersection_id": int})
features = pd.read_csv(DATA / "intersection_features.csv",
                       dtype={"intersection_id": int})
cv       = pd.read_csv(DATA / "cv_scores_full_network.csv",
                       dtype={"intersection_id": int})

print(f"crash_counts.csv          : {len(crashes):>5} rows")
print(f"intersection_features.csv : {len(features):>5} rows")
print(f"cv_scores_full_network.csv: {len(cv):>5} rows")

crash_counts.csv          :    22 rows
intersection_features.csv :    22 rows
cv_scores_full_network.csv:    50 rows


## 2. Assertions and Join

Assertions are checked **before** the join so errors are attributed to the correct source.
A second set of assertions runs after the join to check the merged result.

In [11]:
# ── Pre-join assertions ───────────────────────────────────────────────────────

# crash_counts
assert crashes["intersection_id"].is_unique,     "crash_counts: duplicate intersection_ids"
assert (crashes["crash_count"] >= 0).all(),     "crash_counts: negative crash counts"
assert (crashes["exposure"] > 0).all(),     "crash_counts: zero or negative exposure"

# intersection_features
assert features["intersection_id"].is_unique,     "intersection_features: duplicate intersection_ids"
assert features["dim_type"].isin(["T", "4+"]).all(),     f"intersection_features: unexpected dim_type values: {features['dim_type'].unique()}"
assert features["dim_priority"].isin(["VRI", "voorrang", "geen_voorrang"]).all(),     f"intersection_features: unexpected dim_priority values: {features['dim_priority'].unique()}"

# cv_scores
assert cv["intersection_id"].is_unique,     "cv_scores: duplicate intersection_ids"
valid_splits = {"train", "val", "unlabelled", "excluded"}
bad_splits = set(cv["cv_split"].unique()) - valid_splits
assert not bad_splits, f"cv_scores: unexpected cv_split values: {bad_splits}"

print("Pre-join assertions passed.")

# ── Inner join ────────────────────────────────────────────────────────────────
# Drop intersections absent from any source; report counts for transparency.
df = crashes.merge(features, on="intersection_id", how="inner")
n_after_features = len(df)
df = df.merge(cv, on="intersection_id", how="inner")
n_final = len(df)

print()
print("Join completeness:")
print(f"  crash_counts                 : {len(crashes):>5} intersections")
print(f"  after merge with features    : {n_after_features:>5} retained "
      f"({len(crashes) - n_after_features} dropped)")
print(f"  after merge with cv_scores   : {n_final:>5} retained "
      f"({n_after_features - n_final} dropped)")
print(f"  Total dropped                : {len(crashes) - n_final:>5}")

# ── Post-join assertions ──────────────────────────────────────────────────────
assert df["intersection_id"].is_unique, "Post-join: duplicate intersection_ids"
assert (df["crash_count"] >= 0).all(),  "Post-join: negative crash counts"
assert (df["exposure"]    >  0).all(),  "Post-join: zero/negative exposure"

print()
print("Post-join assertions passed.")
print()
print(df["cv_split"].value_counts().rename("cv_split distribution").to_string())

Pre-join assertions passed.

Join completeness:
  crash_counts                 :    22 intersections
  after merge with features    :    22 retained (0 dropped)
  after merge with cv_scores   :    22 retained (0 dropped)
  Total dropped                :     0

Post-join assertions passed.

cv_split
unlabelled    22


## 3. Flag CV Split Boundary

Intersections used to **train or validate the CV model** must be excluded from
the NB modelling dataset. We flag them here rather than dropping, so downstream
notebooks can decide when to apply the filter and assert it explicitly.

The flag column `cv_split_flagged` is True where `cv_split` is `"train"` or `"val"`.

In [12]:
# Flag train/val rows — these are excluded in NB03 and NB04
df["cv_split_flagged"] = df["cv_split"].isin(["train", "val"])

n_flagged   = df["cv_split_flagged"].sum()
n_modelling = (~df["cv_split_flagged"]).sum()

print(f"Flagged (train/val)   : {n_flagged:>5}  — excluded from NB03/NB04")
print(f"Modelling set (remainder): {n_modelling:>5}  — used to fit NB models")

Flagged (train/val)   :     0  — excluded from NB03/NB04
Modelling set (remainder):    22  — used to fit NB models


## 4. Missing CV Scores

Not all intersections in the network have a CV-predicted score.
Reasons include: no panoramic photo within range, photo quality filter, or
intersection excluded from reprojection.

**Proposal Section 4.4.4** specifies two strategies:

1. **Drop (default):** Remove intersections with `cv_score_predicted = NaN` from
   Phase 4 only. Phase 3 (baseline NB) is fitted on the full modelling set.
   This is the primary analysis dataset.

2. **Mean imputation (sensitivity):** Replace missing scores with the column mean
   and add a binary indicator `has_cv_score`. Saved separately as
   `nb_model_ready_imputed.csv` for sensitivity checks in NB04.

Both datasets are produced here. NB04 loads the primary version by default.

In [13]:
# Count and report missing CV scores
n_missing_cv = df["cv_score_predicted"].isna().sum()
n_has_cv     = df["cv_score_predicted"].notna().sum()

print(f"cv_score_predicted present : {n_has_cv:>5}")
print(f"cv_score_predicted missing : {n_missing_cv:>5}")
print(f"Missing share              : {n_missing_cv / len(df):.1%}")
print()

# Binary indicator (used in imputation sensitivity dataset)
df["has_cv_score"] = df["cv_score_predicted"].notna().astype(int)

# Sensitivity dataset: mean imputation
df_imputed = df.copy()
cv_mean = df_imputed["cv_score_predicted"].mean()
df_imputed["cv_score_predicted"] = df_imputed["cv_score_predicted"].fillna(cv_mean)
print(f"Imputation mean (cv_score): {cv_mean:.4f}")

cv_score_predicted present :    22
cv_score_predicted missing :     0
Missing share              : 0.0%

Imputation mean (cv_score): 0.6271


## 5. Categorical Encoding

`dim_type` and `dim_priority` are nominal categoricals. We use one-hot (dummy)
encoding with explicit reference category choices:

| Variable | Reference category | Rationale |
|----------|-------------------|-----------|
| `dim_type` | `T` (3-arm) | T-intersections are the most common type |
| `dim_priority` | `geen_voorrang` (uncontrolled) | Baseline — no explicit priority rule |

The reference dummies (`dim_type_T`, `dim_priority_geen_voorrang`) are dropped
to avoid perfect multicollinearity. The retained dummies are:
- `dim_type_4p` (1 = 4-or-more-arm intersection)
- `dim_priority_VRI` (1 = traffic-light controlled)
- `dim_priority_voorrang` (1 = priority road, no lights)

Note: `4+` is renamed to `dim_type_4p` (replacing `+` with `p`) because the
`+` character causes issues in model formula strings.

In [14]:
# One-hot encode dim_type and dim_priority
# Rename "4+" to "4p" before encoding to avoid formula-string issues
df["dim_type"] = df["dim_type"].str.replace("+", "p", regex=False)
df_imputed["dim_type"] = df_imputed["dim_type"].str.replace("+", "p", regex=False)

def encode_categoricals(frame: pd.DataFrame) -> pd.DataFrame:
    '''One-hot encode dim_type and dim_priority; drop reference dummies.

    Reference categories: dim_type=T, dim_priority=geen_voorrang.
    Proposal Section 4.4.2: dummy coding for nominal predictors.
    '''
    out = pd.get_dummies(
        frame,
        columns=["dim_type", "dim_priority"],
        drop_first=False,   # create all dummies, drop references manually below
        dtype=int,
    )
    # Drop the reference dummies
    reference_cols = ["dim_type_T", "dim_priority_geen_voorrang"]
    out.drop(columns=[c for c in reference_cols if c in out.columns], inplace=True)
    return out


df        = encode_categoricals(df)
df_imputed = encode_categoricals(df_imputed)

# Report dummy columns created
dummy_cols = [c for c in df.columns
              if c.startswith("dim_type_") or c.startswith("dim_priority_")]
print("Dummy columns created (reference categories dropped):")
for col in dummy_cols:
    print(f"  {col:<35}  mean = {df[col].mean():.3f}")

Dummy columns created (reference categories dropped):
  dim_type_4p                          mean = 0.500
  dim_priority_VRI                     mean = 0.364
  dim_priority_voorrang                mean = 0.318


In [ ]:
# ── Save primary dataset ──────────────────────────────────────────────────────
out_path = PROCESSED / "nb_model_ready.csv"
df.to_csv(out_path, index=False)
size_kb = out_path.stat().st_size / 1024
print(f"Saved nb_model_ready.csv        ({size_kb:.1f} KB, {len(df)} rows, {df.shape[1]} columns)")

# ── Save imputation sensitivity dataset ───────────────────────────────────────
imp_path = PROCESSED / "nb_model_ready_imputed.csv"
df_imputed.to_csv(imp_path, index=False)
size_kb_imp = imp_path.stat().st_size / 1024
print(f"Saved nb_model_ready_imputed.csv ({size_kb_imp:.1f} KB, {len(df_imputed)} rows)")

## 6. Dataset Summary

Final check: confirm the output is complete and self-consistent.

In [16]:
# Summary table printed at end of NB01 for a quick sanity check
print("=" * 55)
print("NB01 DATASET SUMMARY — nb_model_ready.csv")
print("=" * 55)

# Row counts
print(f"  Total intersections          : {len(df):>5}")
print(f"  Flagged (train/val)          : {df['cv_split_flagged'].sum():>5}")
print(f"  Modelling set (not flagged)  : {(~df['cv_split_flagged']).sum():>5}")
print(f"  Has CV score                 : {df['has_cv_score'].sum():>5}")
print(f"  Missing CV score             : {(1 - df['has_cv_score']).sum():>5}")
print()

# Crash count distribution
cc = df["crash_count"]
print("  Crash count distribution:")
print(f"    Min / Max                  : {cc.min()} / {cc.max()}")
print(f"    Mean                       : {cc.mean():.3f}")
print(f"    Variance                   : {cc.var():.3f}")
print(f"    Variance/Mean (>1 -> OD)   : {cc.var() / cc.mean():.3f}")
print(f"    Zero-crash share           : {(cc == 0).mean():.1%}")
print()

# Column inventory
print(f"  Columns in output ({df.shape[1]} total):")
for col in df.columns:
    dtype = str(df[col].dtype)
    n_miss = df[col].isna().sum()
    miss_str = f"  [{n_miss} missing]" if n_miss > 0 else ""
    print(f"    {col:<35} {dtype:<10}{miss_str}")

print()
print("NB01 complete. Next: run NB02_eda.ipynb")

NB01 DATASET SUMMARY — nb_model_ready.csv
  Total intersections          :    22
  Flagged (train/val)          :     0
  Modelling set (not flagged)  :    22
  Has CV score                 :    22
  Missing CV score             :     0

  Crash count distribution:
    Min / Max                  : 0 / 6
    Mean                       : 1.727
    Variance                   : 2.874
    Variance/Mean (>1 -> OD)   : 1.664
    Zero-crash share           : 27.3%

  Columns in output (14 total):
    intersection_id                     int32     
    crash_count                         int64     
    exposure                            int64     
    n_legs                              int64     
    speed_limit                         int64     
    intensity_major                     int64     
    intensity_minor                     int64     
    cv_score_predicted                  float64   
    cv_split                            str       
    cv_split_flagged                    bool   

## Handover

**Produced by this notebook:**

| File | Description |
|------|-------------|
| `data/processed/nb_model_ready.csv` | Primary dataset — missing CV scores dropped at point of use in NB04 |
| `data/processed/nb_model_ready_imputed.csv` | Sensitivity dataset — missing CV scores mean-imputed |

**Column contract for downstream notebooks:**

| Column | Type | Notes |
|--------|------|-------|
| `intersection_id` | int | Join key throughout |
| `crash_count` | int | Injury crashes 2019-2024 |
| `exposure` | float | Denominator for crash rate offset |
| `cv_score_predicted` | float / NaN | NaN where no CV score available |
| `cv_split` | str | Original split label |
| `cv_split_flagged` | bool | True = train or val (exclude from NB model) |
| `has_cv_score` | int | 1 = CV score present |
| `n_legs`, `speed_limit`, `intensity_major`, `intensity_minor` | float | Continuous predictors |
| `dim_type_4p` | int | 1 = 4-or-more-arm (ref: T) |
| `dim_priority_VRI` | int | 1 = traffic-light (ref: geen_voorrang) |
| `dim_priority_voorrang` | int | 1 = priority road (ref: geen_voorrang) |

**What NB02 expects:** Load `data/processed/nb_model_ready.csv`; all columns above present.
**What NB03 expects:** Same file; apply `~cv_split_flagged` filter on load.
**What NB04 expects:** Same file + `data/processed/model_baseline.pkl` from NB03.